# Practical Session 8: Support Vector Machines and Margin-Based Classification

This notebook follows the current SVM practical and explains hard-margin and
soft-margin modeling, support vectors, the dual view, and classifier comparison.


In [ ]:
# Import NumPy for geometric calculations and matrix operations.
import numpy as np
# Import pandas for tabular model summaries.
import pandas as pd
# Import cvxpy for the primal and dual SVM optimization problems.
import cvxpy as cp
# Import Matplotlib for geometric visualizations.
import matplotlib.pyplot as plt
# Import logistic regression for the final classifier comparison.
from sklearn.linear_model import LogisticRegression

# Keep printed arrays readable.
np.set_printoptions(precision=4, suppress=True)
# Select an available solver for the convex programs.
SOLVER = "CLARABEL" if "CLARABEL" in cp.installed_solvers() else "SCS"

# Store names for annotations and summary tables.
names = np.array(["Mueller", "Kroos", "Reus", "Gomez", "Goetze"])
# Store the two-dimensional feature vectors.
X = np.array(
    [
        [10.0, 0.1],
        [2.0, 0.7],
        [6.0, 0.6],
        [8.0, 0.1],
        [8.0, 0.4],
    ]
)
# Store the labels of the four training samples in {-1, +1}.
y = np.array([1, -1, 1, -1], dtype=float)
# Split training data and the new sample.
X_train = X[:4]
x_goetze = X[4]


## Task 1: Data inspection and geometric intuition

The first plot shows the geometry of the four labeled samples in the original feature
plane.


In [ ]:
# Plot the positive class points.
plt.figure(figsize=(6, 4))
plt.scatter(X_train[y == 1, 0], X_train[y == 1, 1], label="Class +1", s=80)
# Plot the negative class points.
plt.scatter(X_train[y == -1, 0], X_train[y == -1, 1], label="Class -1", s=80, marker="x")
# Annotate each training sample with its name.
for idx, name in enumerate(names[:4]):
    plt.annotate(name, (X_train[idx, 0], X_train[idx, 1]), textcoords="offset points", xytext=(5, 5))
plt.xlabel("Goals")
plt.ylabel("PCR value")
plt.title("Original SVM training data")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Task 2: Hard-margin SVM formulation

We solve the standard primal hard-margin problem and inspect whether the current data
admits a separating hyperplane with margin at least one in the normalized constraints.


In [ ]:
def solve_hard_margin_svm(X, y):
    # Create the separating hyperplane parameters.
    w = cp.Variable(X.shape[1])
    b = cp.Variable()
    # Enforce y_i (w^T x_i + b) >= 1 for all samples.
    # TODO: Add the hard-margin constraints.
    constraints = [...]
    # Minimize one half of the squared weight norm.
    # TODO: Build the hard-margin SVM problem.
    problem = ...
    problem.solve(solver=SOLVER)
    return problem, w, b


# Solve the hard-margin model on the original data.
hard_problem, hard_w, hard_b = solve_hard_margin_svm(X_train, y)
print("Hard-margin status on original data:", hard_problem.status)
if hard_problem.status in {"optimal", "optimal_inaccurate"}:
    # Convert the weight vector into a geometric margin.
    margin = 1.0 / np.linalg.norm(hard_w.value)
    print("Margin:", margin)
    print("Parameters:", hard_w.value, hard_b.value)
else:
    print("The hard-margin problem is infeasible on the original data.")


## Task 3: Slight data modification for a clearer hard-margin separation

We change one feature value slightly so the geometry becomes easier to visualize and the
resulting hard-margin separator has a somewhat clearer margin.


In [ ]:
# Copy the original training set and modify Gomez's PCR value slightly.
X_modified = X_train.copy()
X_modified[3, 1] = 0.05

# Solve the hard-margin model on the modified data set.
hard_problem_mod, hard_w_mod, hard_b_mod = solve_hard_margin_svm(X_modified, y)
hard_margin = 1.0 / np.linalg.norm(hard_w_mod.value)
# The support vectors satisfy the margin constraint with equality.
margins = y * (X_modified @ hard_w_mod.value + hard_b_mod.value)
support_vectors = np.where(np.isclose(margins, 1.0, atol=1e-4))[0]

print("Modified hard-margin status:", hard_problem_mod.status)
print("Modified hard-margin parameters:", hard_w_mod.value, hard_b_mod.value)
print("Geometric margin:", hard_margin)
print("Support vector indices:", support_vectors)


In [ ]:
def plot_svm(ax, X, y, w, b, title, support_vector_idx=None):
    # Plot the two classes with different markers.
    ax.scatter(X[y == 1, 0], X[y == 1, 1], label="Class +1", s=80)
    ax.scatter(X[y == -1, 0], X[y == -1, 1], label="Class -1", s=80, marker="x")
    # Create x-values for the decision and margin lines.
    x_vals = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200)
    if abs(w[1]) > 1e-12:
        # Draw the separating hyperplane w^T x + b = 0.
        decision = -(b + w[0] * x_vals) / w[1]
        # Draw the two margin boundaries w^T x + b = ±1.
        margin_plus = -(b - 1 + w[0] * x_vals) / w[1]
        margin_minus = -(b + 1 + w[0] * x_vals) / w[1]
        ax.plot(x_vals, decision, color="black", label="Decision boundary")
        ax.plot(x_vals, margin_plus, color="gray", linestyle="--", label="Margins")
        ax.plot(x_vals, margin_minus, color="gray", linestyle="--")
    if support_vector_idx is not None and len(support_vector_idx) > 0:
        # Highlight support vectors by circles without face color.
        ax.scatter(
            X[support_vector_idx, 0],
            X[support_vector_idx, 1],
            facecolors="none",
            edgecolors="red",
            s=180,
            linewidths=2,
            label="Support vectors",
        )
    ax.set_xlabel("Goals")
    ax.set_ylabel("PCR value")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)


# Plot the modified hard-margin separator and its support vectors.
fig, ax = plt.subplots(figsize=(6, 4))
plot_svm(ax, X_modified, y, hard_w_mod.value, hard_b_mod.value, "Modified hard-margin SVM", support_vectors)
plt.show()


## Task 4: Soft-margin SVM on the original data

The soft-margin model adds slack variables and therefore stays useful even when the data
is difficult or impossible to separate perfectly.


In [ ]:
def solve_soft_margin_svm(X, y, C):
    # Create the hyperplane parameters and one slack variable per sample.
    w = cp.Variable(X.shape[1])
    b = cp.Variable()
    xi = cp.Variable(len(y), nonneg=True)
    # Relax the margin constraints by the nonnegative slacks xi_i.
    # TODO: Add the soft-margin constraints with slack variables.
    constraints = [...]
    # Minimize norm regularization plus C times the sum of slacks.
    problem = cp.Problem(cp.Minimize(0.5 * cp.sum_squares(w) + C * cp.sum(xi)), constraints)
    problem.solve(solver=SOLVER)
    return problem, w, b, xi


# Test several regularization strengths C.
C_values = [0.2, 1.0, 10.0]
soft_rows = []
soft_models = {}
for C in C_values:
    soft_problem, soft_w, soft_b, soft_xi = solve_soft_margin_svm(X_train, y, C)
    # Compute signed decision values and convert them into hard labels.
    decision = X_train @ soft_w.value + soft_b.value
    predictions = np.sign(decision)
    predictions[predictions == 0] = 1
    accuracy = np.mean(predictions == y)
    soft_rows.append(
        {
            "C": C,
            "status": soft_problem.status,
            "weights": np.round(soft_w.value, 4).tolist(),
            "bias": float(soft_b.value),
            "slack_variables": np.round(soft_xi.value, 4).tolist(),
            "training_accuracy": accuracy,
        }
    )
    soft_models[C] = (soft_w.value, float(soft_b.value), soft_xi.value)

# Display the effect of C on the learned classifier.
display(pd.DataFrame(soft_rows))


In [ ]:
# Plot the soft-margin separators for all tested C values side by side.
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, C in zip(axes, C_values):
    w_c, b_c, _ = soft_models[C]
    plot_svm(ax, X_train, y, w_c, b_c, f"Soft-margin SVM (C={C})")
plt.tight_layout()
plt.show()


## Task 5: Dual view and support vectors

We solve the dual problem for `C = 1` and reconstruct the primal weight vector from the
dual coefficients.


In [ ]:
# Use C = 1 for the explicit dual reconstruction task.
C_dual = 1.0
# Compute the linear kernel matrix X X^T.
K = X_train @ X_train.T
# Form the quadratic matrix diag(y) K diag(y).
Q = np.outer(y, y) * K
# Create one dual variable alpha_i per sample.
alpha = cp.Variable(len(y))
dual_problem = cp.Problem(
    cp.Maximize(cp.sum(alpha) - 0.5 * cp.quad_form(alpha, Q)),
    [alpha >= 0, alpha <= C_dual, y @ alpha == 0],
)
dual_problem.solve(solver=SOLVER)

# Extract the dual coefficients and reconstruct the primal weight vector.
alpha_value = alpha.value
# TODO: Reconstruct the primal weight vector from the dual variables.
reconstructed_w = ...
primal_w, primal_b, _ = soft_models[C_dual]
support_idx_dual = np.where(alpha_value > 1e-5)[0]

# Report the support vectors identified by nonzero dual coefficients.
dual_df = pd.DataFrame(
    {"alpha": alpha_value, "is_support_vector": alpha_value > 1e-5},
    index=names[:4],
)
display(dual_df)
print("Reconstructed primal weight vector:", reconstructed_w)
print("Direct primal weight vector:", primal_w)
print("Difference:", reconstructed_w - primal_w)


## Task 6: Classification of the new sample

We classify Goetze with both the modified hard-margin model and the original
soft-margin model and report signed distances to the corresponding hyperplanes.


In [ ]:
def signed_distance(x, w, b):
    # Normalize the raw score by ||w|| to obtain the geometric signed distance.
    return (x @ w + b) / np.linalg.norm(w)


# Compute the raw scores of the new sample for both trained models.
goetze_hard_score = x_goetze @ hard_w_mod.value + hard_b_mod.value
goetze_soft_score = x_goetze @ primal_w + primal_b

# Summarize hard predictions, distances, and margin-region membership.
classification_df = pd.DataFrame(
    [
        {
            "model": "modified hard margin",
            "predicted_class": int(np.sign(goetze_hard_score) or 1),
            "signed_distance": signed_distance(x_goetze, hard_w_mod.value, hard_b_mod.value),
            "inside_margin_region": abs(goetze_hard_score) <= 1,
        },
        {
            "model": "original soft margin",
            "predicted_class": int(np.sign(goetze_soft_score) or 1),
            "signed_distance": signed_distance(x_goetze, primal_w, primal_b),
            "inside_margin_region": abs(goetze_soft_score) <= 1,
        },
    ]
)
display(classification_df)


## Task 7: Comparison across classifiers

We finish by comparing perceptron, logistic regression, and soft-margin SVM on the same
sample to highlight their different outputs and optimization objectives.


In [ ]:
def perceptron_classifier(X, y, alpha=0.5, max_epochs=100):
    # Augment the samples by a bias column.
    X_aug = np.hstack([np.ones((X.shape[0], 1)), X])
    # Start from zero weights.
    w = np.zeros(X_aug.shape[1])
    for _ in range(max_epochs):
        errors = 0
        for i in range(len(y)):
            # Apply the perceptron update whenever the current point is misclassified.
            if y[i] * np.dot(w, X_aug[i]) <= 0:
                w += alpha * y[i] * X_aug[i]
                errors += 1
        if errors == 0:
            break
    return w


# Train the perceptron on the original four-point data set.
perceptron_w = perceptron_classifier(X_train, y)
perceptron_score = np.dot(np.r_[1.0, x_goetze], perceptron_w)

# Train a logistic-regression model for comparison of outputs.
logistic_model = LogisticRegression(C=1e6, solver="lbfgs", max_iter=5000)
# TODO: Fit the logistic-regression comparison model.
...
logistic_class = int(logistic_model.predict(x_goetze.reshape(1, -1))[0])
logistic_probability = float(logistic_model.predict_proba(x_goetze.reshape(1, -1))[0, 1])

# Summarize the conceptual differences across the three classifiers.
classifier_comparison = pd.DataFrame(
    [
        {
            "classifier": "Perceptron",
            "optimization_objective": "classification updates until convergence",
            "output_type": "hard class label",
            "robust_to_nonseparable_data": "no",
            "Goetze_prediction": int(np.sign(perceptron_score) or 1),
        },
        {
            "classifier": "Logistic regression",
            "optimization_objective": "cross-entropy minimization",
            "output_type": "probability and class label",
            "robust_to_nonseparable_data": "yes",
            "Goetze_prediction": logistic_class,
        },
        {
            "classifier": "Soft-margin SVM",
            "optimization_objective": "margin maximization with slack penalty",
            "output_type": "signed margin score and class label",
            "robust_to_nonseparable_data": "yes",
            "Goetze_prediction": int(np.sign(goetze_soft_score) or 1),
        },
    ]
)
display(classifier_comparison)
print("Logistic probability for Goetze being class 1:", logistic_probability)
